# Indexes: GSI and LSI

## Overview
DynamoDB provides two types of secondary indexes that allow queries beyond the primary key.

**Global Secondary Index (GSI):**
- A completely separate index with its own partition key and optional sort key
- Can be created on any attribute(s) at any time (even after table creation)
- Queries are **eventually consistent** only
- Has its own capacity (separate from the base table in Provisioned mode)
- Up to 20 per table

**Local Secondary Index (LSI):**
- Same partition key as the base table, but a **different sort key**
- Must be created at table creation — **cannot be added later**
- Queries can be **strongly consistent** (same partition, same server)
- Shares the base table's capacity
- Limited to 10 GB per partition key value
- Up to 5 per table

**When you need an index:**
- Access pattern requires querying by an attribute that is not the partition key
- You need to sort results by a different attribute than the base sort key
- You want to support multiple entity-type queries efficiently in a single-table design

**Projection types (what data is stored in the index):**

| Type | Stored in index | Use when |
|---|---|---|
| `KEYS_ONLY` | PK, SK, index PK, index SK only | You only need to check existence or count |
| `INCLUDE` | Keys + specified attributes | You need a few specific attributes alongside the keys |
| `ALL` | Every attribute | You need any attribute and don't mind higher storage cost |

---

In [1]:

print("=== GSI vs LSI comparison ===")
comparison = [
    ("Partition key",    "Any attribute",             "Same as base table's PK"),
    ("Sort key",         "Any attribute (optional)",  "Different attribute from base table's SK"),
    ("Created when",     "Any time (even after table creation)", "Only at table creation"),
    ("Consistency",      "Eventually consistent only", "Strongly consistent reads supported"),
    ("Capacity",         "Separate from base table",  "Shares base table's provisioned capacity"),
    ("Item projection",  "KEYS_ONLY, INCLUDE, ALL",   "KEYS_ONLY, INCLUDE, ALL"),
    ("Limit per table",  "20 GSIs (default, can request increase)", "5 LSIs"),
    ("Table size limit", "No extra limit",            "10 GB per partition key value"),
    ("Best for",         "New access patterns not covered by base key", "Same PK, alternate sort order"),
]
print(f"  {'Feature':<20s}  {'GSI':<42s}  LSI")
print("  " + "-"*90)
for feature, gsi, lsi in comparison:
    print(f"  {feature:<20s}  {gsi:<42s}  {lsi}")


=== GSI vs LSI comparison ===
  Feature               GSI                                         LSI
  ------------------------------------------------------------------------------------------
  Partition key         Any attribute                               Same as base table's PK
  Sort key              Any attribute (optional)                    Different attribute from base table's SK
  Created when          Any time (even after table creation)        Only at table creation
  Consistency           Eventually consistent only                  Strongly consistent reads supported
  Capacity              Separate from base table                    Shares base table's provisioned capacity
  Item projection       KEYS_ONLY, INCLUDE, ALL                     KEYS_ONLY, INCLUDE, ALL
  Limit per table       20 GSIs (default, can request increase)     5 LSIs
  Table size limit      No extra limit                              10 GB per partition key value
  Best for              New access 

---
## Creating a GSI

In [2]:

print("=== Creating a GSI: query observations by species ===")
gsi_create = '''
# Problem: the base table key is (site_id PK, obs_sk SK).
# Access pattern: "give me all observations of Atlantic Salmon (SP#SALMON)
#                  across ALL sites, sorted by date"
# Solution: GSI with species_id as PK, obs_sk (date) as SK

# Add GSI at table creation (or use update_table for an existing table):
table = dynamodb.create_table(
    TableName="EcoObservations",
    KeySchema=[
        {"AttributeName": "site_id",   "KeyType": "HASH"},
        {"AttributeName": "obs_sk",    "KeyType": "RANGE"},
    ],
    AttributeDefinitions=[
        {"AttributeName": "site_id",   "AttributeType": "S"},
        {"AttributeName": "obs_sk",    "AttributeType": "S"},
        {"AttributeName": "species_id","AttributeType": "S"},  # GSI key
    ],
    GlobalSecondaryIndexes=[{
        "IndexName": "species-date-index",
        "KeySchema": [
            {"AttributeName": "species_id", "KeyType": "HASH"},
            {"AttributeName": "obs_sk",     "KeyType": "RANGE"},
        ],
        "Projection": {"ProjectionType": "INCLUDE",
                       "NonKeyAttributes": ["site_id", "common_name", "count", "at_risk"]},
        "BillingMode": "PAY_PER_REQUEST",
    }],
    BillingMode="PAY_PER_REQUEST",
)

# ── Add a GSI to an existing table ──────────────────────────────
table.update(
    AttributeDefinitions=[
        {"AttributeName": "region", "AttributeType": "S"},
    ],
    GlobalSecondaryIndexUpdates=[{
        "Create": {
            "IndexName": "region-date-index",
            "KeySchema": [
                {"AttributeName": "region", "KeyType": "HASH"},
                {"AttributeName": "obs_sk", "KeyType": "RANGE"},
            ],
            "Projection": {"ProjectionType": "ALL"},
        }
    }],
)
# GSI creation is online and non-blocking; table remains fully available
# Building time depends on table size; new writes are indexed immediately
'''
print(gsi_create)


=== Creating a GSI: query observations by species ===

# Problem: the base table key is (site_id PK, obs_sk SK).
# Access pattern: "give me all observations of Atlantic Salmon (SP#SALMON)
#                  across ALL sites, sorted by date"
# Solution: GSI with species_id as PK, obs_sk (date) as SK

# Add GSI at table creation (or use update_table for an existing table):
table = dynamodb.create_table(
    TableName="EcoObservations",
    KeySchema=[
        {"AttributeName": "site_id",   "KeyType": "HASH"},
        {"AttributeName": "obs_sk",    "KeyType": "RANGE"},
    ],
    AttributeDefinitions=[
        {"AttributeName": "site_id",   "AttributeType": "S"},
        {"AttributeName": "obs_sk",    "AttributeType": "S"},
        {"AttributeName": "species_id","AttributeType": "S"},  # GSI key
    ],
    GlobalSecondaryIndexes=[{
        "IndexName": "species-date-index",
        "KeySchema": [
            {"AttributeName": "species_id", "KeyType": "HASH"},
            {"AttributeName":

---
## Querying a GSI and LSI example

In [3]:

print("=== Querying a GSI ===")
gsi_query = '''
# Query the species-date-index GSI:
# "All Atlantic Salmon observations in 2023, most recent first"
response = table.query(
    IndexName="species-date-index",
    KeyConditionExpression=(
        Key("species_id").eq("SP#SALMON") &
        Key("obs_sk").begins_with("OBS#2023")
    ),
    FilterExpression=Attr("at_risk").eq(True),
    ScanIndexForward=False,      # descending by date
    Limit=50,
)
items = response["Items"]

# ── Sparse GSI: only index items that have the attribute ─────────
# If species_id is only present on observation items (not metadata),
# the GSI only indexes those items -- a built-in sparse index.
# Items without species_id are simply not in the index.
# This is much cheaper than indexing every item in the table.

# ── GSI projection types ─────────────────────────────────────────
# KEYS_ONLY:  index stores only PK, SK, GSI PK, GSI SK (smallest, cheapest)
# INCLUDE:    index also stores specified extra attributes (targeted)
# ALL:        index stores all item attributes (largest, most expensive)
#
# Rule: project only attributes you actually read from the index.
# Fetching non-projected attributes requires a read of the base table (extra cost).
'''
print(gsi_query)

print("LSI example (same PK, different SK):")
lsi_example = '''
# Base table: site_id (PK), obs_sk (SK) = "OBS#2023-04-10#001"
# LSI:        site_id (PK), species_id (SK)
# Allows: "all observations at site 001 sorted by species_id" without a GSI

# Must be declared at table creation — cannot be added later:
table = dynamodb.create_table(
    TableName="EcoObservations",
    KeySchema=[
        {"AttributeName": "site_id",   "KeyType": "HASH"},
        {"AttributeName": "obs_sk",    "KeyType": "RANGE"},
    ],
    AttributeDefinitions=[
        {"AttributeName": "site_id",   "AttributeType": "S"},
        {"AttributeName": "obs_sk",    "AttributeType": "S"},
        {"AttributeName": "species_id","AttributeType": "S"},
    ],
    LocalSecondaryIndexes=[{
        "IndexName": "site-species-index",
        "KeySchema": [
            {"AttributeName": "site_id",    "KeyType": "HASH"},
            {"AttributeName": "species_id", "KeyType": "RANGE"},
        ],
        "Projection": {"ProjectionType": "INCLUDE",
                       "NonKeyAttributes": ["obs_sk", "count", "life_stage"]},
    }],
    BillingMode="PAY_PER_REQUEST",
)
'''
print(lsi_example)


=== Querying a GSI ===

# Query the species-date-index GSI:
# "All Atlantic Salmon observations in 2023, most recent first"
response = table.query(
    IndexName="species-date-index",
    KeyConditionExpression=(
        Key("species_id").eq("SP#SALMON") &
        Key("obs_sk").begins_with("OBS#2023")
    ),
    FilterExpression=Attr("at_risk").eq(True),
    ScanIndexForward=False,      # descending by date
    Limit=50,
)
items = response["Items"]

# ── Sparse GSI: only index items that have the attribute ─────────
# If species_id is only present on observation items (not metadata),
# the GSI only indexes those items -- a built-in sparse index.
# Items without species_id are simply not in the index.
# This is much cheaper than indexing every item in the table.

# ── GSI projection types ─────────────────────────────────────────
# KEYS_ONLY:  index stores only PK, SK, GSI PK, GSI SK (smallest, cheapest)
# INCLUDE:    index also stores specified extra attributes (targeted)
# ALL:       

---
## Common Pitfalls

**1. Not planning LSIs at table creation**
LSIs cannot be added after a table is created. If you realise 6 months into production that you need an LSI, you must create a new table, migrate all data, and switch over. Always think through all access patterns before creating a table, and add LSIs upfront for any case where you need the same partition key with a different sort order.

**2. Over-projecting with `ALL` on every GSI**
`ProjectionType: ALL` copies every attribute into the index. A GSI with ALL projection on a table with wide items costs nearly as much storage as the base table itself (times the number of such GSIs). Project only the attributes actually read from that index. If a query needs non-projected attributes, DynamoDB fetches them from the base table -- adding latency and RCU cost.

**3. Hot GSI partitions from low-cardinality GSI partition keys**
A GSI with `region` as the PK and only 5 distinct regions concentrates traffic on 5 GSI partitions. If 90% of queries are for `region = 'Atlantic'`, that partition is a hot spot. Add a `shard_id` suffix to the GSI key for high-traffic values, or redesign to distribute load.

**4. Expecting GSI reads to be strongly consistent**
GSI reads are always eventually consistent. The GSI is asynchronously updated after the base table write. A GetItem immediately followed by a Query on a GSI for the same item may return stale data. If strong consistency is required, Query the base table directly.

**5. Exceeding the 10 GB LSI partition limit**
Each LSI partition (items sharing the same partition key value) is limited to 10 GB. A patient with thousands of encounters in a healthcare table could hit this limit. Plan partition sizes carefully and use GSIs for unbounded relationships.

**6. Creating GSIs reactively without access pattern analysis**
Each GSI adds write latency (the index must be updated on every write), storage cost, and operational complexity. A table with 10 GSIs is expensive to write to. Audit access patterns first and create GSIs only for confirmed, frequent query patterns -- not speculatively.


---
*sql_methods_library - Samantha McGarrigle*